# Batch call costs: last night's session

Loads every `raw_responses/*.jsonl` file from the most recent run under `data/runs/` (one file per batch call), flattens the `usage` block reported by the LLM gateway for every request, and totals it per batch call:
- overall cost and token counts
- token breakdown (reasoning, cached, cache-write, ...)
- cost breakdown (input, output, cached input, cache write, web search, request fee, ...)

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from foodscraper.config import *

plt.style.use("seaborn-v0_8-pastel")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [2]:
run_dir = sorted(RUNS_DIR.iterdir())[-1]
raw_responses_dir = run_dir / "raw_responses"
files = sorted(raw_responses_dir.glob("*.jsonl"))

run_dir, len(files)

(PosixPath('/Users/rakul/Desktop/cs/5sem/msc/foodscraps/data/runs/2026-09-17'),
 4)

## Load every request's `usage` block

One row per request; nested dicts (`completion_tokens_details`, `prompt_tokens_details`, `cost_details`) are flattened into `<parent>_<field>` columns.

In [3]:
records = []
for f in files:
    with f.open() as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            flat = pd.json_normalize(record["usage"], sep="_").iloc[0].to_dict()
            flat["file"] = f.name
            # group by the model we asked for, not `record["model"]` (the model actually
            # served) -- the gateway can silently route a single batch call's requests
            # across multiple providers (e.g. google-vertex vs. google-ai-studio) even
            # though they're all one logical batch call
            flat["model"] = record["metadata"]["requested_model"]
            flat["served_model"] = record["model"]
            records.append(flat)

usage_df = pd.DataFrame(records)
usage_df.head()

,completion_tokens,prompt_tokens,total_tokens,reasoning_tokens,cost,completion_tokens_details_accepted_prediction_tokens,completion_tokens_details_audio_tokens,completion_tokens_details_reasoning_tokens,completion_tokens_details_rejected_prediction_tokens,completion_tokens_details_text_tokens,completion_tokens_details_image_tokens,prompt_tokens_details_audio_tokens,prompt_tokens_details_cache_write_tokens,prompt_tokens_details_cached_tokens,prompt_tokens_details_image_tokens,prompt_tokens_details_text_tokens,prompt_tokens_details_video_tokens,cost_details_upstream_inference_cost,cost_details_upstream_inference_prompt_cost,cost_details_upstream_inference_completions_cost,cost_details_total_cost,cost_details_input_cost,cost_details_output_cost,cost_details_cached_input_cost,cost_details_cache_write_input_cost,cost_details_request_cost,cost_details_web_search_cost,cost_details_image_input_cost,cost_details_image_output_cost,cost_details_audio_input_cost,file,model,served_model,prompt_tokens_details_cache_creation_tokens
0,6787,422164,428951,3342,1.132198,None,0,3342,None,None,0,0,0,0,0,None,0,0.912198,0.844328,0.06787,1.132198,0.844328,0.06787,0.0,0.0,0,0.22,None,None,None,prompt_20260910.md__claude-sonnet-5.jsonl,anthropic/claude-sonnet-5,anthropic/claude-sonnet-5,NaN
1,5477,528754,534231,1690,1.342278,None,0,1690,None,None,0,0,0,0,0,None,0,1.112278,1.057508,0.05477,1.342278,1.057508,0.05477,0.0,0.0,0,0.23,None,None,None,prompt_20260910.md__claude-sonnet-5.jsonl,anthropic/claude-sonnet-5,anthropic/claude-sonnet-5,NaN
2,9156,689256,698412,5519,1.710072,None,0,5519,None,None,0,0,0,0,0,None,0,1.470072,1.378512,0.09156,1.710072,1.378512,0.09156,0.0,0.0,0,0.24,None,None,None,prompt_20260910.md__claude-sonnet-5.jsonl,anthropic/claude-sonnet-5,anthropic/claude-sonnet-5,NaN
3,7346,500708,508054,3612,1.314876,None,0,3612,None,None,0,0,0,0,0,None,0,1.074876,1.001416,0.07346,1.314876,1.001416,0.07346,0.0,0.0,0,0.24,None,None,None,prompt_20260910.md__claude-sonnet-5.jsonl,anthropic/claude-sonnet-5,anthropic/claude-sonnet-5,NaN
4,9081,711779,720860,4953,1.804368,None,0,4953,None,None,0,0,0,0,0,None,0,1.514368,1.423558,0.09081,1.804368,1.423558,0.09081,0.0,0.0,0,0.29,None,None,None,prompt_20260910.md__claude-sonnet-5.jsonl,anthropic/claude-sonnet-5,anthropic/claude-sonnet-5,NaN


## Cost per batch call

In [4]:
costs = (
    usage_df.groupby(["model"])
    .agg(
        n_calls=("cost", "count"),
        prompt_tokens=("prompt_tokens", "sum"),
        completion_tokens=("completion_tokens", "sum"),
        total_tokens=("total_tokens", "sum"),
        total_cost=("cost", "sum"),
    )
    .sort_values("total_cost", ascending=False)
    .reset_index()
)
costs


,model,n_calls,prompt_tokens,completion_tokens,total_tokens,total_cost
0,anthropic/claude-sonnet-5,22,11654632,147968,11802600,29.838944
1,openai/gpt-5.6-terra,22,2292207,101790,2393997,8.526132
2,google-ai-studio/gemini-3.7-flash,22,110352,153867,264219,3.862064
3,openai/gpt-5.6-luna,22,1871179,73225,1944404,2.832598


## Token breakdown per batch call

Where the prompt and completion tokens actually went (reasoning, cached, cache-write, image/audio/video, ...).

In [13]:
def drop_empty_cols(df, keep=("model", "file")):
    """Drop columns that are 0/None/NaN in every row (unused by this provider)."""
    has_value = (df.fillna(0) != 0).any(axis=0) | df.columns.isin(keep)
    return df.loc[:, has_value]


token_detail_cols = [
    c
    for c in usage_df.columns
    if c.startswith("completion_tokens_details_")
    or c.startswith("prompt_tokens_details_")
    or c == "reasoning_tokens"
]

token_breakdown = (
    usage_df.groupby(["model"])[token_detail_cols]
    .sum(min_count=1)
    .reset_index()
    .sort_values('reasoning_tokens', ascending=False)
)
token_breakdown = drop_empty_cols(token_breakdown)
token_breakdown

,model,reasoning_tokens,completion_tokens_details_reasoning_tokens,prompt_tokens_details_cache_write_tokens,prompt_tokens_details_cached_tokens,prompt_tokens_details_cache_creation_tokens
1,google-ai-studio/gemini-3.7-flash,117992,117992,0,67706,NaN
3,openai/gpt-5.6-terra,75262,75262,35562,109746,35562.0
0,anthropic/claude-sonnet-5,67636,67636,0,0,NaN
2,openai/gpt-5.6-luna,43914,43914,28903,116405,28903.0


## Cost breakdown per batch call

`cost_details_total_cost` should match `total_cost` above; the other columns show how it's made up (input vs. output tokens, cached/cache-write discounts, web search, per-request fee, ...).

In [6]:
cost_detail_cols = [c for c in usage_df.columns if c.startswith("cost_details_")]

cost_breakdown = (
    usage_df.groupby(["model"])[cost_detail_cols]
    .sum(min_count=1)
    .reset_index()
)
cost_breakdown = drop_empty_cols(cost_breakdown)
cost_breakdown

,model,cost_details_upstream_inference_cost,cost_details_upstream_inference_prompt_cost,cost_details_upstream_inference_completions_cost,cost_details_total_cost,cost_details_input_cost,cost_details_output_cost,cost_details_cached_input_cost,cost_details_cache_write_input_cost,cost_details_web_search_cost
0,anthropic/claude-sonnet-5,24.788944,23.309264,1.479680,29.838944,23.309264,1.479680,0.000000,0.000000,5.050
1,google-ai-studio/gemini-3.7-flash,0.614064,0.037062,0.577001,3.862064,0.031984,0.577001,0.005078,0.000000,3.248
2,openai/gpt-5.6-luna,0.442598,0.354728,0.087870,2.832598,0.345174,0.087870,0.002328,0.007226,2.390
3,openai/gpt-5.6-terra,5.626132,4.404652,1.221480,8.526132,4.293798,1.221480,0.021949,0.088905,2.900


In [7]:
print(f"Total cost across all batch calls: ${costs['total_cost'].sum():.4f}")

Total cost across all batch calls: $45.0597
